# การทดสอบความเร็ว Local LLM (Speed Benchmark: Tokens Per Second)
**หลักสูตร:** Mastering Local AI Stack 2026 (Jupyter Notebook × LM Studio × ComfyUI)  
**วัตถุประสงค์:** วัดความเร็วการสร้างข้อความจริงของโมเดลบนเครื่องของคุณ โดยแยกสถิติ **Time to First Token (TTFT)** และ **Token Generation Speed (tok/s)** ตามหลักวิศวกรรม AI พร้อมเปรียบเทียบกับขีดจำกัดทางทฤษฎีของ Memory Bandwidth

## 1. ทฤษฎี: ทำไมความเร็วของ LLM ถึงวัดด้วย Memory Bandwidth?

การทำงานของโมเดล Large Language Model (LLM) แบ่งออกเป็น 2 ช่วงที่มีคอขวดต่างกันอย่างสิ้นเชิง:

1. **ช่วงที่ 1: Prompt Processing / Prefill (Time to First Token - TTFT)**
   - โมเดลอ่านข้อความคำถามของผู้ใช้พร้อมกันทั้งหมดเพื่อแปลงเป็นเวกเตอร์
   - ประมวลผลแบบขนาน (Matrix-Matrix Multiplication) คอขวดอยู่ที่ **พลังการคำนวณ (Compute / TFLOPS)**
   - ความเร็วช่วงนี้มักสูงมาก (ระดับ 300 – 800+ tokens/วินาที)

2. **ช่วงที่ 2: Token Generation / Decode (Inter-Token Latency)**
   - โมเดลต้องคิดคำตอบทีละคำต่อท้ายไปเรื่อยๆ (Autoregressive)
   - **ทุกครั้งที่สร้างคำศัพท์ใหม่ 1 คำ การ์ดจอต้องโหลดค่าน้ำหนักของโมเดลทั้งหมด (เช่น 2.1 GB) จาก VRAM ผ่าน Memory Bus เข้ามาคำนวณในคอร์ 1 รอบเต็มๆ!**
   - คอขวดช่วงนี้จึงอยู่ที่ **อัตราการส่งข้อมูลของแรมการ์ดจอ (Memory Bandwidth)** ไม่ใช่ความเร็วของชิป GPU

> **สูตรคำนวณทางทฤษฎี:**
> $$\text{ความเร็วสูงสุด (tokens/s)} \approx \frac{\text{Memory Bandwidth (GB/s)}}{\text{ขนาดโมเดลในหน่วยความจำ (GB)}} \times \text{Efficiency } (\approx 70\% - 75\%)\$$

### ตัวเลขสเปกเครื่องในห้องอบรม:
- **NVIDIA GeForce RTX 3050 6GB GDDR6:**
  - Bus Width: 96-bit | Memory Clock: 14 Gbps
  - $\text{Memory Bandwidth} = (96 / 8) \times 14 = \mathbf{168\text{ GB/s}}$
  - รันโมเดล `Qwen2.5-3B (Q4_K_M ~ 2.1 GB)`: ขีดจำกัดทฤษฎี = $168 / 2.1 = 80\text{ tok/s}$
  - **ความเร็วที่คาดหวังในชีวิตจริง (75% Efficiency): $\approx 55 - 65\text{ tokens/s}$**

- **Apple Silicon Mac (Unified Memory):**
  - M1: 68 GB/s ($\approx 25 - 30\text{ tok/s}$)
  - M2 / M3: 100 GB/s ($\approx 35 - 40\text{ tok/s}$)
  - M4: 120 GB/s ($\approx 40 - 50\text{ tok/s}$)
  - M-Pro / M-Max: 150 – 400 GB/s ($\approx 65 - 120+\text{ tok/s}$)

## 2. เชื่อมต่อ Local Server ของ LM Studio
ตรวจสอบว่าคุณได้เปิดโปรแกรม **LM Studio** และกดปุ่ม **Start Server** ที่พอร์ต `1234` เรียบร้อยแล้ว

In [ ]:
import time
import pandas as pd
from openai import OpenAI

# เชื่อมต่อ OpenAI-Compatible API ของ LM Studio
client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

# ตรวจสอบโมเดลที่โหลดไว้ในหน่วยความจำ
try:
    models_list = client.models.list()
    if models_list.data:
        active_model_id = models_list.data[0].id
        print(f"✅ เชื่อมต่อ LM Studio สำเร็จ!")
        print(f"📦 โมเดลที่กำลังทำงาน: {active_model_id}")
    else:
        active_model_id = "local-model"
        print("⚠️ เชื่อมต่อได้ แต่ยังไม่พบโมเดลที่โหลดใน LM Studio (กรุณาโหลดโมเดลก่อน)")
except Exception as e:
    active_model_id = "local-model"
    print(f"❌ ไม่สามารถเชื่อมต่อ LM Studio ได้: {e}")
    print("กรุณาเปิด LM Studio -> เมนู Developer (<->) -> กด Start Server ที่พอร์ต 1234")

## 3. ฟังก์ชันวัดความเร็วแบบ Streaming (วิธีที่แม่นยำ 100%)

ทำไมการวัดแบบปกติ (`start = time.time()` แล้วรอจบ) ถึงผิดพลาด?
- เพราะเวลาที่ได้จะนำเวลาคิดคำถาม (Prefill) + Network Overhead มารวมด้วย ทำให้ตัวเลขความเร็วต่ำกว่าจริง

ฟังก์ชัน `benchmark_generation_speed` ด้านล่างจะยิงคำขอแบบ `stream=True`:
1. จับเวลา `t0` ตอนเริ่มส่งคำขอ
2. บันทึก `t_first` เมื่อได้รับตัวอักษรตัวแรก -> คำนวณเป็น **Time to First Token (TTFT)**
3. นับจำนวน Token ที่พ่นออกมาทีละตัวจนจบ -> คำนวณเป็น **Generation Speed (tok/s)** เฉพาะช่วงสร้างข้อความล้วนๆ

In [ ]:
def benchmark_generation_speed(prompt: str, label: str = "การทดสอบ", max_tokens: int = 250):
    """
    ฟังก์ชันวัดความเร็วการสร้างข้อความของ LLM อย่างแม่นยำด้วยการสตรีมทีละ Token
    """
    print(f"\n{=*65}")
    print(f"🏃 เริ่มต้นทดสอบ: {label}")
    print(f"📝 คำถาม: {prompt}")
    print(f"{=*65}")
    
    t_start = time.time()
    first_token_time = None
    token_chunks = 0
    full_response = []
    
    # เรียกใช้งานแบบสตรีมมิง
    stream = client.chat.completions.create(
        model=active_model_id,
        messages=[
            {"role": "system", "content": "You are a helpful and concise assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=max_tokens,
        stream=True
    )
    
    # อ่าน Token ทีละก้อนที่ถูกพ่นออกมา
    for chunk in stream:
        if chunk.choices and len(chunk.choices) > 0:
            content = chunk.choices[0].delta.content or ""
            if content:
                if first_token_time is None:
                    first_token_time = time.time()  # ได้เวลาคำแรก Time to First Token
                token_chunks += 1
                full_response.append(content)
                print(content, end="", flush=True)
                
    t_end = time.time()
    
    if first_token_time is None:
        print("\n⚠️ โมเดลไม่ได้ตอบข้อความใดๆ ออกมา")
        return None
        
    # คำนวณสถิติ
    ttft = first_token_time - t_start                    # เวลาอ่านโจทย์ (วินาที)
    decode_time = t_end - first_token_time               # เวลาพ่นข้อความ (วินาที)
    total_time = t_end - t_start                         # เวลารวมทั้งหมด (วินาที)
    speed_tok_s = token_chunks / decode_time if decode_time > 0 else 0
    
    print(f"\n\n{-*65}")
    print(f"📊 ผลการทดสอบ: {label}")
    print(f"• Time to First Token (TTFT): {ttft:.3f} วินาที (เวลาทำความเข้าใจโจทย์)")
    print(f"• ระยะเวลาสร้างข้อความ (Decode Time): {decode_time:.3f} วินาที")
    print(f"• จำนวน Token ที่สร้างออกมา: ~{token_chunks} tokens")
    print(f"• 🚀 ความเร็วการสร้างคำตอบเฉลี่ย: {speed_tok_s:.2f} tokens/วินาที")
    print(f"• เวลารวมตั้งแต่ส่งจนจบ: {total_time:.3f} วินาที")
    print(f"{-*65}")
    
    return {
        "การทดสอบ": label,
        "TTFT (วิ)": round(ttft, 3),
        "เวลาสร้างข้อความ (วิ)": round(decode_time, 3),
        "จำนวน Tokens": token_chunks,
        "ความเร็ว (tok/s)": round(speed_tok_s, 2)
    }
print("✅ โหลดฟังก์ชัน benchmark_generation_speed เรียบร้อย!")

## 4. ลงมือทดสอบ 2 รูปแบบ (โจทย์สั้น vs โจทย์ยาว)

สังเกตปรากฏการณ์จริง:
- เมื่อคำถามยาวขึ้น -> **TTFT จะเพิ่มขึ้น** เล็กน้อย (เพราะโมเดลต้องอ่านโจทย์มากขึ้น)
- แต่ **ความเร็วในการสร้างข้อความ (tok/s) จะคงที่** เสมอ เพราะถูกล็อกด้วย Memory Bandwidth ของฮาร์ดแวร์!

In [ ]:
benchmark_results = []

# การทดสอบที่ 1: โจทย์สั้น (งานเขียนคำบรรยายภาพสั้นๆ สำหรับ ComfyUI)
res1 = benchmark_generation_speed(
    prompt="จงแต่งคำบรรยายภาพ Stable Diffusion เป็นภาษาอังกฤษ สไตล์ Cyberpunk Street Market ไม่เกิน 30 คำ",
    label="โจทย์ที่ 1: สั่งแต่ง Image Prompt สั้นๆ",
    max_tokens=80
)
if res1:
    benchmark_results.append(res1)

In [ ]:
# การทดสอบที่ 2: โจทย์ยาว (งานเขียนบทความวิเคราะห์ข้อดีของ Local AI Stack)
res2 = benchmark_generation_speed(
    prompt="จงอธิบายข้อดีของการรัน Local AI บนฮาร์ดแวร์ตัวเอง เทียบกับการใช้ Cloud API 3 ข้อ โดยอธิบายแต่ละข้อให้เห็นภาพชัดเจน",
    label="โจทย์ที่ 2: ตอบคำถามและอธิบายเหตุผลอย่างละเอียด",
    max_tokens=250
)
if res2:
    benchmark_results.append(res2)

## 5. สรุปผลการทดสอบและเปรียบเทียบกับขีดจำกัดฮาร์ดแวร์

In [ ]:
if benchmark_results:
    df_bench = pd.DataFrame(benchmark_results)
    avg_speed = df_bench["ความเร็ว (tok/s)"].mean()
    
    display(df_bench)
    
    print(f"\n🎯 สรุปผลความเร็วเฉลี่ยของเครื่องคุณ: {avg_speed:.2f} tokens/วินาที")
    
    # ตรวจสอบกับเกณฑ์ฮาร์ดแวร์
    if avg_speed >= 50:
        print("🟢 ยอดเยี่ยมมาก! ความเร็วอยู่ในเกณฑ์มาตรฐานของการ์ดจอ RTX 3050 (168 GB/s) หรือชิป Apple Silicon รุ่น Pro/Max")
    elif avg_speed >= 30:
        print("🟡 ความเร็วดี! สอดคล้องกับชิป Apple Silicon Base (M1/M2/M3) หรือมีการแบ่งประมวลผล (Offload) ไปยัง CPU บางส่วน")
    else:
        print("🔵 ความเร็วอยู่ในเกณฑ์ปกติของการรันบน CPU เป็นหลัก (หรือใช้โมเดลขนาด 7B ขึ้นไป)")
else:
    print("ยังไม่มีผลการทดสอบ กรุณาเปิด LM Studio Server แล้วรันเซลล์ด้านบน")